In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/protein_features.parquet')
print("Loaded:", df.shape)
print("Range:", df["date"].min().date(), "to", df["date"].max().date())

# 16-day forecast horizon, mirroring the original competition design
TEST_START  = "2017-07-31"
VALID_START = "2017-07-15"

train = df[df["date"] <  VALID_START].copy()
valid = df[(df["date"] >= VALID_START) & (df["date"] < TEST_START)].copy()
test  = df[df["date"] >= TEST_START].copy()

print(f"\nTrain: {len(train):,} rows | {train['date'].min().date()} to {train['date'].max().date()}")
print(f"Valid: {len(valid):,} rows | {valid['date'].min().date()} to {valid['date'].max().date()}")
print(f"Test:  {len(test):,} rows | {test['date'].min().date()} to {test['date'].max().date()}")
print(f"\nSplit %: {len(train)/len(df)*100:.1f} / {len(valid)/len(df)*100:.1f} / {len(test)/len(df)*100:.1f}")

Loaded: (7459611, 36)
Range: 2015-01-29 to 2017-08-15

Train: 7,205,999 rows | 2015-01-29 to 2017-07-14
Valid: 129,573 rows | 2017-07-15 to 2017-07-30
Test:  124,039 rows | 2017-07-31 to 2017-08-15

Split %: 96.6 / 1.7 / 1.7


In [3]:
# metrics

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def wape(y_true, y_pred):
    """Weighted Absolute Percentage Error — total error / total actual."""
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100

def rmsle(y_true, y_pred):
    """Root Mean Squared Log Error — penalizes under-forecasting more than over."""
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

def evaluate(name, y_true, y_pred, results):
    results.append({
        "model": name,
        "MAE":   round(mae(y_true, y_pred), 4),
        "RMSE":  round(rmse(y_true, y_pred), 4),
        "WAPE":  round(wape(y_true, y_pred), 2),
        "RMSLE": round(rmsle(y_true, y_pred), 4),
    })
    return results

In [4]:
# baselines

results = []
y_valid = valid["unit_sales"].values

# 1. Naive — yesterday's sales
results = evaluate("Naive (lag_1)", y_valid, valid["lag_1"].values, results)

# 2. Seasonal naive — same weekday last week
results = evaluate("Seasonal naive (lag_7)", y_valid, valid["lag_7"].values, results)

# 3. Moving averages
results = evaluate("Moving avg 7d", y_valid, valid["roll_mean_7"].values, results)
results = evaluate("Moving avg 28d", y_valid, valid["roll_mean_28"].values, results)

# 4. Zero baseline — sanity floor
results = evaluate("Always zero", y_valid, np.zeros(len(valid)), results)

baseline_df = pd.DataFrame(results).sort_values("WAPE")
print(baseline_df.to_string(index=False))

                 model    MAE    RMSE       WAPE  RMSLE
         Moving avg 7d 3.9725 13.6616  49.950001 0.7042
        Moving avg 28d 3.9751 13.5639  49.990002 0.7073
Seasonal naive (lag_7) 4.2901 10.2457  53.950001 0.8342
         Naive (lag_1) 5.0158 21.1070  63.070000 0.8568
           Always zero 7.9522 21.1992 100.000000 1.8665
